# Лабораторная работа 7

Квазилинейное уравнение переноса

Решается уравнение

$$
\frac{\partial U}{\partial t} + U \frac{\partial U}{\partial x} = 0
$$

Вариант 21:

$$
U(0, x)=
\begin{cases}
1, & x < 0.5,\\
0, & x \ge 0.5.
\end{cases}
$$

Область: $x \in [0, 1]$, $t \in [0, 1]$. Требуемая точность: $0.01$.


In [1]:
import contextlib
import io
import json
from pathlib import Path

import matplotlib
import numpy as np
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt
from matplotlib import pyplot as plt

matplotlib.use("Agg")

BASE_DIR = Path(__file__).resolve().parent
NOTEBOOK_PATH = BASE_DIR / "7.ipynb"
DOCX_PATH = BASE_DIR / "N_Туманов.docx"
PLOTS_DIR = BASE_DIR / "plots"

X0 = 0.0
X1 = 1.0
T0 = 0.0
T1 = 1.0
EPS = 0.01
GRID_SEQUENCE = [20, 40, 80, 160, 320, 640]
FINAL_N = 640

LEFT_STATE = 1.0
RIGHT_STATE = 0.0
SHOCK_SPEED = 0.5 * (LEFT_STATE + RIGHT_STATE)


def flux(u):
    return 0.5 * u * u


def initial_condition(x):
    return np.where(x < 0.5, LEFT_STATE, RIGHT_STATE)


def exact_solution(x, t):
    shock = 0.5 + SHOCK_SPEED * t
    return np.where(x < shock, LEFT_STATE, RIGHT_STATE)


def right_boundary(t):
    return RIGHT_STATE


def solve_artificial_viscosity(n):
    m = int(round(2.5 * n))
    h = (X1 - X0) / n
    tau = (T1 - T0) / m
    x = np.linspace(X0, X1, n + 1)
    t = np.linspace(T0, T1, m + 1)
    u = np.zeros((m + 1, n + 1))
    u[0, :] = initial_condition(x)

    for layer in range(m + 1):
        u[layer, 0] = LEFT_STATE
        u[layer, -1] = right_boundary(t[layer])

    for layer in range(m):
        u[layer + 1, 0] = LEFT_STATE
        u[layer + 1, -1] = right_boundary(t[layer + 1])
        for i in range(1, n):
            u[layer + 1, i] = (
                0.5 * (u[layer, i + 1] + u[layer, i - 1])
                - tau / (2 * h) * (flux(u[layer, i + 1]) - flux(u[layer, i - 1]))
            )

    return {
        "name": "Метод с искусственной вязкостью",
        "n": n,
        "m": m,
        "h": h,
        "tau": tau,
        "x": x,
        "t": t,
        "u": u,
    }


def solve_conservative(n):
    m = int(round(2.5 * n))
    h = (X1 - X0) / n
    tau = (T1 - T0) / m
    x = np.linspace(X0, X1, n + 1)
    t = np.linspace(T0, T1, m + 1)
    u = np.zeros((m + 1, n + 1))
    u[0, :] = initial_condition(x)

    for layer in range(m + 1):
        u[layer, 0] = LEFT_STATE
        u[layer, -1] = right_boundary(t[layer])

    for layer in range(m):
        u[layer + 1, 0] = LEFT_STATE
        u[layer + 1, -1] = right_boundary(t[layer + 1])
        for i in range(1, n):
            u[layer + 1, i] = u[layer, i] - tau / h * (
                flux(u[layer, i]) - flux(u[layer, i - 1])
            )

    return {
        "name": "Консервативная схема",
        "n": n,
        "m": m,
        "h": h,
        "tau": tau,
        "x": x,
        "t": t,
        "u": u,
    }


def convergence_l1(coarse, fine):
    max_diff = 0.0
    for layer in range(coarse["m"] + 1):
        diff = np.abs(coarse["u"][layer, :] - fine["u"][2 * layer, ::2])
        max_diff = max(max_diff, float(np.mean(diff)))
    return max_diff


def max_l1_error(solution):
    max_err = 0.0
    for layer, tt in enumerate(solution["t"]):
        exact = exact_solution(solution["x"], tt)
        max_err = max(max_err, float(np.mean(np.abs(solution["u"][layer, :] - exact))))
    return max_err


def estimate_shock_position(solution, target_time=0.25):
    layer = int(round(target_time / solution["tau"]))
    values = solution["u"][layer, :]
    x = solution["x"]
    mid = 0.5 * (LEFT_STATE + RIGHT_STATE)
    for i in range(len(x) - 1):
        if (values[i] - mid) * (values[i + 1] - mid) <= 0 and abs(values[i + 1] - values[i]) > 1e-12:
            return float(x[i] + (mid - values[i]) * (x[i + 1] - x[i]) / (values[i + 1] - values[i]))
    return float("nan")


def build_convergence_table(solver):
    rows = []
    previous = None
    for n in GRID_SEQUENCE:
        current = solver(n)
        delta = None if previous is None else convergence_l1(previous, current)
        rows.append(
            {
                "n": current["n"],
                "m": current["m"],
                "h": current["h"],
                "tau": current["tau"],
                "delta": delta,
            }
        )
        previous = current
    return rows


def plot_profile(solution, target_time, filename):
    layer = int(round(target_time / solution["tau"]))
    x = solution["x"]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(x, solution["u"][layer, :], linewidth=2, label="Численное решение")
    ax.plot(x, exact_solution(x, target_time), "k--", linewidth=2, label="Точное решение")
    ax.axvline(0.5 + SHOCK_SPEED * target_time, color="red", linestyle=":", label="Положение разрыва")
    ax.set_xlabel("x")
    ax.set_ylabel("U")
    ax.set_title(f"{solution['name']}, t = {target_time}")
    ax.grid(True, alpha=0.35)
    ax.legend()
    ax.set_ylim(-0.1, 1.1)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.close(fig)
    return PLOTS_DIR / filename


def plot_comparison(visc, cons, target_time, filename):
    layer_v = int(round(target_time / visc["tau"]))
    layer_c = int(round(target_time / cons["tau"]))
    x = visc["x"]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(visc["x"], visc["u"][layer_v, :], linewidth=2, label="Искусственная вязкость")
    ax.plot(cons["x"], cons["u"][layer_c, :], linewidth=2, label="Консервативная схема")
    ax.plot(x, exact_solution(x, target_time), "k--", linewidth=2, label="Точное решение")
    ax.axvline(0.5 + SHOCK_SPEED * target_time, color="red", linestyle=":", label="Точный разрыв")
    ax.set_xlabel("x")
    ax.set_ylabel("U")
    ax.set_title(f"Сравнение профилей при t = {target_time}")
    ax.grid(True, alpha=0.35)
    ax.legend()
    ax.set_xlim(0.55, 0.75)
    ax.set_ylim(-0.1, 1.1)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.close(fig)
    return PLOTS_DIR / filename


def plot_heatmap(solution, filename):
    x = solution["x"]
    t = solution["t"]
    u = solution["u"]
    x_mesh, t_mesh = np.meshgrid(x, t)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    im = ax.pcolormesh(x_mesh, t_mesh, u, shading="auto", cmap="viridis", vmin=0, vmax=1)
    ax.plot(0.5 + SHOCK_SPEED * t, t, "r--", linewidth=2, label="Точный разрыв")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_title(solution["name"])
    ax.legend()
    fig.colorbar(im, ax=ax, label="U")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.close(fig)
    return PLOTS_DIR / filename


def plot_3d(solution, filename):
    x = solution["x"]
    t = solution["t"]
    u = solution["u"]
    x_mesh, t_mesh = np.meshgrid(x, t)
    step_x = max(1, len(x) // 90)
    step_t = max(1, len(t) // 90)
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(
        x_mesh[::step_t, ::step_x],
        t_mesh[::step_t, ::step_x],
        u[::step_t, ::step_x],
        cmap="viridis",
        edgecolor="none",
    )
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_zlabel("U")
    ax.set_title(solution["name"])
    ax.view_init(elev=25, azim=-60)
    fig.colorbar(surf, shrink=0.55, aspect=12)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.close(fig)
    return PLOTS_DIR / filename


def create_plots(visc, cons):
    PLOTS_DIR.mkdir(exist_ok=True)
    return [
        plot_profile(visc, 0.25, "profile_viscosity_t025.png"),
        plot_profile(cons, 0.25, "profile_conservative_t025.png"),
        plot_comparison(visc, cons, 0.25, "profile_comparison_t025.png"),
        plot_heatmap(visc, "heatmap_viscosity.png"),
        plot_heatmap(cons, "heatmap_conservative.png"),
        plot_3d(visc, "surface_viscosity.png"),
        plot_3d(cons, "surface_conservative.png"),
    ]


def format_delta(value):
    return "-" if value is None else f"{value:.6f}"


def print_table(rows):
    print(f"{'N':>6} {'M':>6} {'h':>12} {'tau':>12} {'Delta L1':>12}")
    for row in rows:
        print(
            f"{row['n']:6d} {row['m']:6d} {row['h']:12.7f} "
            f"{row['tau']:12.7f} {format_delta(row['delta']):>12}"
        )



In [2]:
visc_table = build_convergence_table(solve_artificial_viscosity)
cons_table = build_convergence_table(solve_conservative)
visc = solve_artificial_viscosity(FINAL_N)
cons = solve_conservative(FINAL_N)

print('Таблица сходимости. Метод с искусственной вязкостью:')
print_table(visc_table)
print()
print('Таблица сходимости. Консервативная схема:')
print_table(cons_table)
print()
print(f"Итоговая сетка: N = {FINAL_N}, M = {visc['m']}")
print(f"Максимальная L1-ошибка, искусственная вязкость: {max_l1_error(visc):.6f}")
print(f"Максимальная L1-ошибка, консервативная схема: {max_l1_error(cons):.6f}")
print(f"Положение разрыва при t = 0.25, точное: {0.5 + SHOCK_SPEED * 0.25:.6f}")
print(f"Положение разрыва при t = 0.25, искусственная вязкость: {estimate_shock_position(visc):.6f}")
print(f"Положение разрыва при t = 0.25, консервативная схема: {estimate_shock_position(cons):.6f}")
plots = create_plots(visc, cons)
print()
print('Сохраненные графики:')
for path in plots:
    print(path)


Таблица сходимости. Метод с искусственной вязкостью:
     N      M            h          tau     Delta L1
    20     50    0.0500000    0.0200000            -
    40    100    0.0250000    0.0100000     0.052124
    80    200    0.0125000    0.0050000     0.035005
   160    400    0.0062500    0.0025000     0.020917
   320    800    0.0031250    0.0012500     0.010849
   640   1600    0.0015625    0.0006250     0.005446

Таблица сходимости. Консервативная схема:
     N      M            h          tau     Delta L1
    20     50    0.0500000    0.0200000            -
    40    100    0.0250000    0.0100000     0.019211
    80    200    0.0125000    0.0050000     0.009853
   160    400    0.0062500    0.0025000     0.004988
   320    800    0.0031250    0.0012500     0.002509
   640   1600    0.0015625    0.0006250     0.001259

Итоговая сетка: N = 640, M = 1600
Максимальная L1-ошибка, искусственная вязкость: 0.005369
Максимальная L1-ошибка, консервативная схема: 0.001309
Положение разры